# Keep the LLM's Answer When It Fails Validation

In this video, I'm going to keep an LLM's whole answer about a photo when it fails a Pydantic model. The error stays on its row, and we'll rerun the LLM only for the photos that failed.

With Pydantic alone, one bad field raises an error for the whole model, and the fields that passed don't come back either. Two threads in the Pydantic repo ask about this: how to keep the valid fields, and how to fall back instead of crashing.

*Show pydantic issue #9371 and discussion #13811.*

We'll see that failure in plain Pydantic first, then put the same answer in a Pixeltable table. The notebook is in the `keep-failed-answers` folder of the repo linked below.

## Start from an empty `answers_demo` directory

Start Jupyter with `uv run jupyter lab` in this folder. When we sent these six photos to Gemini, every answer fit our Pydantic model, so the LLM calls here go to a stand-in that breaks two of those answers. It needs no API key.

In [ ]:
!uv sync

In [ ]:
import pixeltable as pxt
from pydantic import ValidationError

import stand_in
from schema import ListingDetails

# Start fresh: drop the tables from any earlier run.
pxt.drop_dir('answers_demo', force=True, if_not_exists='ignore')

## One wrong field fails the whole Pydantic model

Here's the Pydantic model the LLM's answer has to fit. All six fields are required, and `category` and `condition` allow only a few values.

In [ ]:
!sed -n '14,22p' schema.py

This is Gemini's answer for a photo of a film camera, from our run, with one change: `color` is a list instead of a string.

In [ ]:
answer = dict(stand_in.ANSWERS['film-camera-1']['answer'], color=['Silver', 'Black'])  # a list, not a string
answer

`model_validate` returns a `ListingDetails`, or raises a `ValidationError`.

In [ ]:
details = None
try:
    details = ListingDetails.model_validate(answer)
except ValidationError as e:
    print(e)
print(details)

It raised for `color`, and `details` is still `None`. The other five fields were fine, and none of them came back.

*Show the maintainer's reply in discussion #13811.*

Pydantic's maintainers suggest wrapping each field in a validator that falls back to a default. That works, but you have to wrap every field and give each one a default.

## The answer and its check go in two columns

Pixeltable is a multimodal backend, and you declare its tables in Python. In `listings`, the only columns you fill in are `name` and `photo`. `response` and `details` are computed columns, so Pixeltable computes them for every row you insert.

In [ ]:
!sed -n '10,12p;25,40p' schema.py

`response` calls the LLM with the prompt and the photo, and asks for JSON in the shape of `ListingDetails`. It keeps the whole answer as it came back.

`details` takes the answer's `parsed` field, the JSON the LLM returned, and checks it against `ListingDetails`: every field there, with the right type. It doesn't run your validators or check `Literal` values, so a `condition` of "barely used" would pass.

`stand_in_gemini` is the demo-only part, with the same arguments as `pxtf.gemini.generate_content`.

In [ ]:
!sed -n '25,41p' stand_in.py

It returns each answer under `parsed`, where Gemini's response puts its JSON. `BROKEN` names the two photos whose answers it breaks, and `CALLS` records every photo it's called for.

`pxt schema diff` shows the plan, and `pxt schema update` creates the table.

In [ ]:
!pxt schema diff schema.py answers_demo

In [ ]:
!pxt schema update schema.py answers_demo -f

## By default, one failed check stores none of the rows

I'll insert six photos of secondhand items. Each `photo` is a URL, and Pixeltable downloads the photo when it inserts the row.

In [ ]:
BASE = 'https://raw.githubusercontent.com/apreshill/pixeltable-marketplace/main/samples/'
NAMES = ['desk-lamp-1', 'film-camera-1', 'guitar-1', 'backpack-1', 'cruiser-bike-1', 'salt-lamp-1']
rows = [{'name': name, 'photo': f'{BASE}{name}.jpg'} for name in NAMES]

listings = pxt.get_table('answers_demo.listings')
try:
    listings.insert(rows)
except pxt.Error as e:
    print(str(e).splitlines()[1])  # the error, without the long expression around it
listings.count()

The film camera's answer failed the check, so the insert stored none of the six rows. With a real LLM, you'd pay for those calls and keep none of the answers.

## With `on_error='ignore'`, every row is stored with its error

`on_error='ignore'` stores every row, and keeps each error on the row it came from.

In [ ]:
listings.insert(rows, on_error='ignore')

Six rows, with two errors, both in `details`. Every computed column has an `errortype` and an `errormsg` you can select.

In [ ]:
listings.select(listings.photo, listings.name, listings.details.errortype, listings.details.errormsg).collect()

The film camera's `color` is a list, and the guitar's answer left out `condition`. The other four answers passed the check.

`failed` is every row where `details` has an error. Here's what those two rows kept.

In [ ]:
failed = listings.details.errortype != None
listings.where(failed).select(
    listings.name,
    listings.response.parsed.title,
    listings.response.parsed.condition,
    listings.response.parsed.color,
    listings.details,
).collect()

`response` still has each whole answer, title included. `details` is empty on these rows, because this works per row, not per field. Pixeltable doesn't keep a partial `ListingDetails`.

## Rerun the LLM call, not the check, on the rows that failed

The errors are in `details`, so the first thing to try is `errors_only=True` on `details`.

In [ ]:
stand_in.CALLS.clear()
listings.recompute_columns(listings.details, errors_only=True).num_rows

In [ ]:
stand_in.CALLS, listings.where(failed).count()

It checked the same two answers again, and both still fail. The stand-in wasn't called at all.

To get new answers, we rerun `response`. `errors_only=True` won't find these rows there, because the LLM call itself worked, so a `where` on `details.errortype` picks them.

With a real LLM, a rerun is a new call, so the answer can come back whole or fail the same way again. The stand-in always breaks the same two answers, so I'll clear its `BROKEN` list first.

In [ ]:
stand_in.BROKEN.clear()  # demo only: the stand-in stops breaking answers
listings.recompute_columns(listings.response, where=failed).num_rows

In [ ]:
stand_in.CALLS

The stand-in was called for the film camera and the guitar, and no other photo. Pixeltable reran `details` after `response`, because `details` is built on it.

In [ ]:
listings.select(listings.photo, listings.name, listings.details.condition, listings.details.color, listings.details.errortype).collect()

Every row passes the check now, and the stand-in was never called again for the four answers that passed the first time.

## For real Gemini calls, `schema_gemini.py` changes the `response` line

This file is the same table, with `pxtf.gemini.generate_content` in place of the stand-in.

In [ ]:
!diff schema.py schema_gemini.py

Set `GEMINI_API_KEY` before you use it.

## Each failed answer stayed on its row

We kept the LLM's whole answer in one column and its check against the Pydantic model in another. With `on_error='ignore'`, the two answers that failed stayed on their rows with their errors, and the rerun called the LLM only for those two.

For big batches, our video Rerun Only the Failures, linked in the description, covers chunked inserts and `pxt errors`.

The notebook and schema files are in the `keep-failed-answers` folder of the repo linked below. If this helps, a star on the Pixeltable GitHub repo helps other developers find it.